# Wyznaczanie przejazdow i punktow OD

Ten notebook jest roboczym szkieletem pod docelowa macierz przejazdow dla fragmentu trasy S8. Pracujemy na danych z indywidualnymi punktami GPS: `data/2024_11_17_17_00_processed.csv`.

Cel pierwszego etapu: wyznaczyc przejazdy oraz wiarygodne punkty poczatkowe i koncowe. Macierz OD powinna byc liczona dopiero po odfiltrowaniu urwanych, niepewnych albo odstajacych trajektorii.

## 0. Plan pracy

Proponowana kolejnosc implementacji:

1. Wczytac i uporzadkowac punkty GPS po `vehicle_id` i czasie.
2. Potraktowac `vehicle_id` jako identyfikator telefonu, nie pojazdu. Jednostka analizy pojazdu/ruchu to pojedynczy przejazd.
3. Policzac cechy miedzy kolejnymi punktami: `time_diff`, `distance_m`, predkosc liczona z GPS, skoki i postoje.
4. Wstepnie podzielic punkty na przejazdy przez przerwy czasowe, skoki GPS i dlugie postoje.
5. Odfiltrowac przejazdy, ktore wygladaja jak rower/pieszy albo ogolnie nie-samochodowy slad.
6. Wyciagnac kandydatow na poczatki i konce przejazdow.
7. Wykryc wiarygodne terminale: klastry endpointow, krance badanego obszaru, okolice POI z OSM.
8. Nazwac klastry endpointow na podstawie najblizszych obiektow OSM i polozenia.
9. Oznaczyc wiarygodnosc kazdego przejazdu i odfiltrowac przejazdy urwane w srodku trasy.
10. Zbudowac pierwsza macierz OD na poziomie wykrytych terminali.

Wazne rozroznienie: w tych danych nie mamy liczby pojazdow na odcinku, tylko slady wybranych telefonow. Mozemy wiec rekonstruowac przejazdy i ich punkty OD, ale nie mierzymy pelnego natezenia ruchu w sensie inzynierii ruchu.


## 1. Importy i parametry

Parametry sa zebrane w jednym miejscu, bo beda wymagały strojenia. Obecne wartosci sa startowe, nie finalne.

In [ ]:

from pathlib import Path

import branca.colormap as cm
import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from shapely import wkt
from sklearn.cluster import DBSCAN

try:
    import osmnx as ox
except ImportError:
    ox = None

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)

DATA_FILE = Path("data/2024_11_17_17_00_processed.csv")
SEGMENTS_FILE = Path("data/data_2024-11-17.csv")
OSM_POI_CACHE = Path("data/external/osm_poi_near_s8.geojson")

PARAMS = {
    "time_gap_new_trip_s": 5 * 60,
    "max_plausible_speed_kmh": 180,
    "min_trip_points": 4,
    "min_trip_duration_s": 30,
    "min_trip_distance_m": 200,
    "stop_speed_kmh": 3,
    "long_stop_s": 3 * 60,
    "endpoint_cluster_eps_m": 150,
    "endpoint_cluster_min_samples": 8,
    "boundary_distance_m": 120,
    "poi_distance_m": 250,
    "osm_bbox_buffer_deg": 0.01,
    "bike_max_reported_speed_kmh": 35,
    "bike_max_gps_speed_kmh": 45,
    "bike_median_speed_kmh": 25,
    "car_min_p90_speed_kmh": 45,
    "car_min_max_speed_kmh": 60,
    "traffic_jam_speed": 10,
    "min_vehicles_traffic": 5,
    "min_slow_frac_traffic": 0.6
}

PARAMS

## 2. Wczytanie danych

`timestamp` wyglada na czas oryginalny, a `time` jest przesuniety o godzine. Do analiz sekwencji uzywamy `timestamp`, bo jest zgodny z nazwa pliku i poprzednim notebookiem.

In [ ]:
df_raw = pd.read_csv(DATA_FILE, parse_dates=["timestamp", "time"])
print(df_raw.head(3))
df_raw.head()

In [ ]:
df_raw.info(memory_usage="deep")

In [ ]:
overview = pd.Series({
    "rows": len(df_raw),
    "vehicles": df_raw["vehicle_id"].nunique(),
    "regions": df_raw["region"].nunique(),
    "timestamp_min": df_raw["timestamp"].min(),
    "timestamp_max": df_raw["timestamp"].max(),
    "time_min": df_raw["time"].min(),
    "time_max": df_raw["time"].max(),
})

overview

## 3. Segmenty referencyjne z danych agregowanych

Pliki `data_YYYY-MM-DD.csv` zawieraja agregaty predkosci na segmentach oraz geometrie `LINESTRING`. Nie daja pojedynczych przejazdow, ale moga sluzyc jako warstwa referencyjna korytarza trasy i ksztaltow odcinkow.


In [ ]:
segments_raw = pd.read_csv(SEGMENTS_FILE, parse_dates=["time"])
segments_raw = segments_raw.rename(columns={"segmnet_id": "segment_id"})

segments_daily = (
    segments_raw
    .groupby(["segment_id", "wkt"], as_index=False)
    .agg(
        avg_speed_day=("avg_speed", "mean"),
        median_speed_day=("avg_speed", "median"),
        observations=("avg_speed", "size"),
    )
)
segments_daily["geometry"] = segments_daily["wkt"].apply(wkt.loads)
segments_gdf = gpd.GeoDataFrame(segments_daily, geometry="geometry", crs="EPSG:4326")

segments_gdf.head()


In [ ]:
segment_map_center = [df_raw["lat"].mean(), df_raw["lon"].mean()]
segment_context_map = folium.Map(location=segment_map_center, zoom_start=12, tiles="CartoDB positron")

speed_min = float(segments_gdf["median_speed_day"].quantile(0.05))
speed_max = float(segments_gdf["median_speed_day"].quantile(0.95))
speed_colormap = cm.LinearColormap(["#d73027", "#fee08b", "#1a9850"], vmin=speed_min, vmax=speed_max)
speed_colormap.caption = "Mediana predkosci segmentu, caly dzien"

for _, row in segments_gdf.iterrows():
    coords = [(lat, lon) for lon, lat in row.geometry.coords]
    speed = float(np.clip(row["median_speed_day"], speed_min, speed_max))
    folium.PolyLine(
        locations=coords,
        color=speed_colormap(speed),
        weight=2,
        opacity=0.75,
        tooltip=(
            f"segment: {row.segment_id}<br>"
            f"mediana predkosci: {row.median_speed_day:.1f}<br>"
            f"obserwacje: {row.observations}"
        ),
    ).add_to(segment_context_map)

sample_points = df_raw.sample(min(len(df_raw), 5000), random_state=42)
for _, row in sample_points.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=1,
        color="black",
        fill=True,
        fill_opacity=0.18,
        opacity=0.18,
    ).add_to(segment_context_map)

speed_colormap.add_to(segment_context_map)
SEGMENT_CONTEXT_MAP_FILE = Path("segment_context_map.html")
segment_context_map.save(SEGMENT_CONTEXT_MAP_FILE)
segment_context_map


## 4. Cechy trajektorii

Liczymy odleglosc miedzy kolejnymi punktami danego pojazdu, predkosc empiryczna z GPS oraz flagi potencjalnie problematycznych obserwacji. To bedzie podstawa do dzielenia na przejazdy i usuwania odstajacych trajektorii.

In [ ]:
def haversine_m(lat1, lon1, lat2, lon2):
    radius_m = 6_371_000
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * radius_m * np.arcsin(np.sqrt(a))


df = df_raw.sort_values(["vehicle_id", "timestamp"]).copy()
df["prev_timestamp"] = df.groupby("vehicle_id")["timestamp"].shift(1)
df["prev_lat"] = df.groupby("vehicle_id")["lat"].shift(1)
df["prev_lon"] = df.groupby("vehicle_id")["lon"].shift(1)

df["time_diff_s"] = (df["timestamp"] - df["prev_timestamp"]).dt.total_seconds()
df["distance_m"] = haversine_m(df["prev_lat"], df["prev_lon"], df["lat"], df["lon"])
df.loc[df["time_diff_s"].isna(), "distance_m"] = np.nan
df["gps_speed_kmh"] = (df["distance_m"] / df["time_diff_s"]) * 3.6

df["is_stationary_point"] = df["speed"].le(PARAMS["stop_speed_kmh"])
df["is_implausible_jump"] = df["gps_speed_kmh"].gt(PARAMS["max_plausible_speed_kmh"])

df[["vehicle_id", "timestamp", "lat", "lon", "speed", "time_diff_s", "distance_m", "gps_speed_kmh",
    "is_implausible_jump"]].head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(df["speed"], bins=50, ax=axes[0])
axes[0].set_title("Raportowana predkosc")

sns.histplot(df["time_diff_s"].dropna().clip(upper=300), bins=50, ax=axes[1])
axes[1].set_title("Przerwy miedzy punktami, obciete do 300 s")

sns.histplot(df["gps_speed_kmh"].dropna().clip(upper=250), bins=50, ax=axes[2])
axes[2].set_title("Predkosc liczona z GPS, obcieta do 250 km/h")

plt.tight_layout()

## 4.5 Korki
Korki oznaczają że wszystkie pojazdy w pobliżu w tym samym czasie również mają prędkości bliskie zeru

### A. Grid based

In [ ]:
# ~100–200 m grid cells (tune as needed)
GRID_SIZE = 0.002  # degrees ≈ 150–200 m

df["lat_bin"] = (df["lat"] / GRID_SIZE).round().astype(int)
df["lon_bin"] = (df["lon"] / GRID_SIZE).round().astype(int)
# traffic is stable over minutes not seconds
df["time_bin"] = df["timestamp"].dt.floor("1min")

# compute local average speed per grid cell per minute
agg = (
    df.groupby(["lat_bin", "lon_bin", "time_bin"])
    .agg(
        mean_speed=("gps_speed_kmh", "mean"),
        median_speed=("gps_speed_kmh", "median"),
        vehicle_count=("vehicle_id", "nunique"),
        slow_fraction=("gps_speed_kmh", lambda x: (x < 10).mean())  # threshold 10 km/h
    )
    .reset_index()
)

In [ ]:
agg["is_traffic_jam"] = (
        (agg["median_speed"] < PARAMS['traffic_jam_speed']) &
        (agg["vehicle_count"] >= PARAMS['min_vehicles_traffic']) &
        (agg["slow_fraction"] >= PARAMS['min_slow_frac_traffic'])
)

In [ ]:
df = df.merge(
    agg[["lat_bin", "lon_bin", "time_bin", "is_traffic_jam"]],
    on=["lat_bin", "lon_bin", "time_bin"],
    how="left"
)

df[["vehicle_id", "timestamp", "lat", "lon", "gps_speed_kmh", "is_traffic_jam"]]

In [ ]:
traffic_cells = agg[agg["is_traffic_jam"]].copy()

traffic_map_center = [df_raw["lat"].mean(), df_raw["lon"].mean()]
traffic_map = folium.Map(location=traffic_map_center, zoom_start=12, tiles="CartoDB positron")

for _, row in traffic_cells.iterrows():
    lat = row["lat_bin"] * GRID_SIZE
    lon = row["lon_bin"] * GRID_SIZE

    bounds = [
        [lat - GRID_SIZE / 2, lon - GRID_SIZE / 2],
        [lat + GRID_SIZE / 2, lon + GRID_SIZE / 2],
    ]

    folium.Rectangle(
        bounds=bounds,
        color="red",
        fill=True,
        fill_opacity=0.35,
        weight=1,
        tooltip=(
            f"Traffic jam<br>"
            f"median speed: {row.median_speed:.1f} km/h<br>"
            f"vehicles: {row.vehicle_count}<br>"
            f"slow fraction: {row.slow_fraction:.2f}"
        ),
    ).add_to(traffic_map)

In [ ]:
sample_points = df[df["is_traffic_jam"]].sample(min(len(df[df["is_traffic_jam"]]), 5000), random_state=42)

for _, row in sample_points.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=2,
        color="red" if row["is_traffic_jam"] else "blue",
        fill=True,
        fill_opacity=0.6,
        opacity=0.6,
    ).add_to(traffic_map)

In [ ]:
traffic_map

### B. Along the streets

In [ ]:
points_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.lon, df.lat),
    crs="EPSG:4326"
)

points_with_segments = gpd.sjoin_nearest(
    points_gdf,
    segments_gdf[["segment_id", "geometry"]],
    how="left",
    distance_col="dist_to_segment"
)
points_with_segments["time_bin"] = points_with_segments["timestamp"].dt.floor("1min")

seg_agg = (
    points_with_segments
    .groupby(["segment_id", "time_bin"])
    .agg(
        median_speed=("gps_speed_kmh", "median"),
        mean_speed=("gps_speed_kmh", "mean"),
        vehicle_count=("vehicle_id", "nunique"),
        slow_fraction=("gps_speed_kmh", lambda x: (x < 10).mean())
    )
    .reset_index()
)
TRAFFIC_SPEED_KMH = 10
MIN_VEHICLES = 5
MIN_SLOW_FRACTION = 0.6

seg_agg["is_traffic_jam"] = (
        (seg_agg["median_speed"] < TRAFFIC_SPEED_KMH) &
        (seg_agg["vehicle_count"] >= MIN_VEHICLES) &
        (seg_agg["slow_fraction"] >= MIN_SLOW_FRACTION)
)
seg_agg_gdf = seg_agg.merge(
    segments_gdf[["segment_id", "geometry"]],
    on="segment_id",
    how="left"
)

seg_agg_gdf = gpd.GeoDataFrame(seg_agg_gdf, geometry="geometry", crs="EPSG:4326")

In [ ]:
seg_agg_gdf["time_bin"] = pd.to_datetime(seg_agg_gdf["time_bin"]).dt.tz_localize(None)

In [ ]:
traffic_map = folium.Map(
    location=[df["lat"].mean(), df["lon"].mean()],
    zoom_start=12,
    tiles="CartoDB positron"
)

for _, row in seg_agg_gdf[seg_agg_gdf["time_bin"] == '2024-11-17T16:22:00'].iterrows():
    coords = [(lat, lon) for lon, lat in row.geometry.coords]
    color = "red" if row.is_traffic_jam else "green"

    folium.PolyLine(
        locations=coords,
        color=color,
        weight=4 if row.is_traffic_jam else 2,
        opacity=0.8,
        tooltip=(
            f"Segment: {row.segment_id}<br>"
            f"Median speed: {row.median_speed:.1f} km/h<br>"
            f"Vehicles: {row.vehicle_count}<br>"
            f"Slow fraction: {row.slow_fraction:.2f}<br>"
            f"Traffic jam: {row.is_traffic_jam}",
            f"Time bin: {row.time_bin}<br>"
        )
    ).add_to(traffic_map)

traffic_map

### C. Time-slider along the streets map

In [ ]:

from folium.plugins import TimestampedGeoJson

features = []
seg_agg_gdf.sort_values(by=["time_bin"], inplace=True)
for _, row in seg_agg_gdf.iterrows():
    coords = [[lon, lat] for lon, lat in row.geometry.coords]

    color = "red" if row.is_traffic_jam else "green"

    feature = {
        "type": "Feature",
        "geometry": {
            "type": "LineString",
            "coordinates": coords,
        },
        "properties": {
            # MUST be a list named "times"
            "times": [row.time_bin.isoformat(), (row.time_bin + pd.Timedelta(minutes=1)).isoformat()],
            "style": {
                "color": color,
                "weight": 4 if row.is_traffic_jam else 2,
                "opacity": 0.8,
            },
            "popup": (
                f"Segment: {row.segment_id}<br>"
                f"Median speed: {row.median_speed:.1f} km/h<br>"
                f"Vehicles: {row.vehicle_count}<br>"
                f"Slow fraction: {row.slow_fraction:.2f}<br>"
                f"Traffic jam: {row.is_traffic_jam}"
            ),
        },
    }

    features.append(feature)

geojson = {
    "type": "FeatureCollection",
    "features": features,
}


In [ ]:
features

In [ ]:
traffic_map = folium.Map(
    location=[df["lat"].mean(), df["lon"].mean()],
    zoom_start=12,
    tiles="CartoDB positron"
)

TimestampedGeoJson(
    data=geojson,
    transition_time=200,
    loop=False,
    auto_play=False,
    add_last_point=False,
    period="PT1M",  # 1-minute bins
    duration="PT1M",  # tells TimeDimension how long features last
    # removePastLayers=True,        # <-- THIS MAKES THEM DISAPPEAR
).add_to(traffic_map)

traffic_map

## 5. Wykrywanie dlugich postojow

Postoj traktujemy ostroznie. Krotkie odcinki z predkoscia bliska zeru moga byc korkiem. Dlugie postoje moga oznaczac koniec jednej podrozy i poczatek kolejnej, szczegolnie jesli wystepuja przy centrum handlowym, wezle lub innym POI.

In [ ]:
df["stationary_block"] = (
    df.groupby("vehicle_id")["is_stationary_point"]
    .transform(lambda s: s.ne(s.shift()).cumsum())
)

stop_blocks = (
    df[df["is_stationary_point"]]
    .groupby(["vehicle_id", "stationary_block"])
    .agg(
        stop_start=("timestamp", "min"),
        stop_end=("timestamp", "max"),
        stop_points=("timestamp", "size"),
        lat=("lat", "mean"),
        lon=("lon", "mean"),
    )
    .reset_index()
)
stop_blocks["stop_duration_s"] = (stop_blocks["stop_end"] - stop_blocks["stop_start"]).dt.total_seconds()
long_stops = stop_blocks[stop_blocks["stop_duration_s"] >= PARAMS["long_stop_s"]].copy()

long_stops.sort_values("stop_duration_s", ascending=False).head(10)

## 6. Wstepny podzial na przejazdy

Nowy przejazd zaczynamy, gdy pojawia sie nowy pojazd, duza przerwa czasowa, niewiarygodny skok GPS albo punkt po dlugim postoju. Ta logika jest celowo parametryczna.

In [ ]:
long_stop_keys = set(zip(long_stops["vehicle_id"], long_stops["stationary_block"]))
df["is_long_stop_block"] = list(zip(df["vehicle_id"], df["stationary_block"])).copy()
df["is_long_stop_block"] = df["is_long_stop_block"].isin(long_stop_keys)
df["prev_is_long_stop_block"] = df.groupby("vehicle_id")["is_long_stop_block"].shift(1, fill_value=False)

df["new_trip_reason"] = "continue"
df.loc[df["time_diff_s"].isna(), "new_trip_reason"] = "first_point"
df.loc[df["time_diff_s"].gt(PARAMS["time_gap_new_trip_s"]), "new_trip_reason"] = "time_gap"
df.loc[df["is_implausible_jump"], "new_trip_reason"] = "gps_jump"
df.loc[df["prev_is_long_stop_block"] & ~df["is_long_stop_block"], "new_trip_reason"] = "after_long_stop"

df["new_trip"] = df["new_trip_reason"].ne("continue")
df["trip_seq"] = df.groupby("vehicle_id")["new_trip"].cumsum()
df["trip_uid"] = df["vehicle_id"].astype(str) + "_" + df["trip_seq"].astype(str)

df["new_trip_reason"].value_counts()

In [ ]:
trip_stats = (
    df.groupby("trip_uid")
    .agg(
        vehicle_id=("vehicle_id", "first"),
        trip_seq=("trip_seq", "first"),
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
        points=("timestamp", "size"),
        start_lat=("lat", "first"),
        start_lon=("lon", "first"),
        end_lat=("lat", "last"),
        end_lon=("lon", "last"),
        origin_region=("region", "first"),
        destination_region=("region", "last"),
        reported_speed_mean=("speed", "mean"),
        gps_speed_max=("gps_speed_kmh", "max"),
        distance_m=("distance_m", "sum"),
        implausible_jumps=("is_implausible_jump", "sum"),
        traffic_jam=("is_traffic_jam", "max"),
    )
    .reset_index()
)
trip_stats["duration_s"] = (trip_stats["end_time"] - trip_stats["start_time"]).dt.total_seconds()

trip_stats["too_few_points"] = trip_stats["points"].lt(PARAMS["min_trip_points"])
trip_stats["too_short_duration"] = trip_stats["duration_s"].lt(PARAMS["min_trip_duration_s"])
trip_stats["too_short_distance"] = trip_stats["distance_m"].lt(PARAMS["min_trip_distance_m"])
trip_stats["has_implausible_jump"] = trip_stats["implausible_jumps"].gt(0)
trip_quality_flags = [
    "too_few_points",
    "too_short_duration",
    "too_short_distance",
    "has_implausible_jump",
]
trip_stats["valid_basic"] = ~trip_stats[trip_quality_flags].any(axis=1)

trip_stats.describe(include="all").T


### Diagnostyka odrzuconych przejazdow

Te flagi odpowiadaja dokladnie na pytania o przejazdy bardzo krotkie czasowo oraz bardzo krotkie dystansowo. Przejazd moze odpasc z kilku powodow naraz, dlatego sumy flag nie musza dawac liczby wszystkich odrzuconych przejazdow.


In [ ]:
temp = trip_stats[trip_stats['valid_basic'] == False].copy()
trip_rejection_summary = pd.Series({
    "all_trips": len(trip_stats),
    "valid_basic": int(trip_stats["valid_basic"].sum()),
    "rejected_basic": int((~trip_stats["valid_basic"]).sum()),
    "too_few_points": int(temp["too_few_points"].sum()),
    "too_short_duration": int(temp["too_short_duration"].sum()),
    "too_short_distance": int(temp["too_short_distance"].sum()),
    "has_implausible_jump": int(temp["has_implausible_jump"].sum()),
    "has_traffic_jam": int(temp["traffic_jam"].sum()),
})

trip_rejection_summary


In [ ]:
rejected_examples = trip_stats.loc[~trip_stats["valid_basic"], [
    "trip_uid",
    "points",
    "duration_s",
    "distance_m",
    "implausible_jumps",
    *trip_quality_flags,
]].sort_values(["too_short_distance", "too_short_duration", "too_few_points"], ascending=False)

rejected_examples.head(20)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(trip_stats["points"], bins=50, ax=axes[0])
axes[0].set_title("Liczba punktow w przejezdzie")
sns.histplot(trip_stats["duration_s"].clip(upper=3600), bins=50, ax=axes[1])
axes[1].set_title("Czas przejazdu, obciety do 1h")
sns.histplot(trip_stats["distance_m"].clip(upper=20_000), bins=50, ax=axes[2])
axes[2].set_title("Dystans GPS, obciety do 20 km")
plt.tight_layout()

## 7. Klasyfikacja przejazdow: samochod vs wolny/rowerowy slad

`vehicle_id` jest identyfikatorem telefonu, wiec nie zakladamy, ze odpowiada stale jednemu pojazdowi. Klasyfikujemy pojedyncze przejazdy. Na razie stosujemy prosta, konserwatywna heurystyke: przejazd jest `car_like`, jesli ma typowo samochodowe predkosci; `bike_or_slow_candidate`, jesli caly przebieg jest wolny; reszta zostaje jako `uncertain_mode`.

To nie jest finalny model. Chodzi o to, zeby na etapie macierzy OD domyslnie liczyc tylko przejazdy samochodopodobne, a wolne slady analizowac osobno.


In [ ]:
df["gps_speed_kmh_for_stats"] = df["gps_speed_kmh"].replace([np.inf, -np.inf], np.nan)

trip_speed_features = (
    df.groupby("trip_uid")
    .agg(
        reported_speed_median=("speed", "median"),
        reported_speed_p90=("speed", lambda s: s.quantile(0.90)),
        reported_speed_max=("speed", "max"),
        gps_speed_median=("gps_speed_kmh_for_stats", "median"),
        gps_speed_p90=("gps_speed_kmh_for_stats", lambda s: s.quantile(0.90)),
        stationary_share=("is_stationary_point", "mean"),
    )
    .reset_index()
)

trip_stats = trip_stats.merge(trip_speed_features, on="trip_uid", how="left")

bike_like = (
        trip_stats["reported_speed_max"].le(PARAMS["bike_max_reported_speed_kmh"])
        & trip_stats["gps_speed_max"].le(PARAMS["bike_max_gps_speed_kmh"])
        & trip_stats["reported_speed_median"].le(PARAMS["bike_median_speed_kmh"])
)
car_like = (
        trip_stats["reported_speed_p90"].ge(PARAMS["car_min_p90_speed_kmh"])
        | trip_stats["reported_speed_max"].ge(PARAMS["car_min_max_speed_kmh"])
)

trip_stats["mode_class"] = np.select(
    [bike_like, car_like],
    ["bike_or_slow_candidate", "car_like"],
    default="uncertain_mode",
)
trip_stats["usable_for_endpoint_detection"] = trip_stats["valid_basic"] & trip_stats["mode_class"].eq("car_like")

trip_stats["mode_class"].value_counts()

In [ ]:
for mode_class in trip_stats["mode_class"].unique():
    temp = trip_stats[trip_stats["mode_class"] == mode_class]
    mode_class_summary = pd.Series({
        "all_trips": len(temp),
        "valid_basic": int(temp["valid_basic"].sum()),
        "rejected_basic": int((~temp["valid_basic"]).sum()),
        "too_few_points": int(temp["too_few_points"].sum()),
        "too_short_duration": int(temp["too_short_duration"].sum()),
        "too_short_distance": int(temp["too_short_distance"].sum()),
        "has_implausible_jump": int(temp["has_implausible_jump"].sum()),
        "has_traffic_jam": int(temp[temp["valid_basic"] == False]["traffic_jam"].sum()),
    })
    print("\n", mode_class)
    print(mode_class_summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

sns.histplot(data=trip_stats, x="reported_speed_median", hue="mode_class", bins=40, multiple="stack", ax=axes[0])
axes[0].set_title("Mediana raportowanej predkosci w przejezdzie")

sns.histplot(data=trip_stats, x="reported_speed_p90", hue="mode_class", bins=40, multiple="stack", ax=axes[1])
axes[1].set_title("90 percentyl raportowanej predkosci")

sns.scatterplot(
    data=trip_stats.sample(min(len(trip_stats), 2000), random_state=42),
    x="reported_speed_median",
    y="reported_speed_max",
    hue="mode_class",
    alpha=0.6,
    ax=axes[2],
)
axes[2].set_title("Wolne slady vs przejazdy samochodopodobne")

plt.tight_layout()


In [ ]:
trip_stats.groupby("mode_class").agg(
    trips=("trip_uid", "size"),
    valid_basic=("valid_basic", "sum"),
    usable_for_endpoint_detection=("usable_for_endpoint_detection", "sum"),
    median_duration_s=("duration_s", "median"),
    median_distance_m=("distance_m", "median"),
    median_reported_speed=("reported_speed_median", "median"),
    median_p90_speed=("reported_speed_p90", "median"),
).sort_values("trips", ascending=False)


## 8. Kandydaci na endpointy

Wyciagamy pierwszy i ostatni punkt kazdego przejazdu. Potem klastrujemy te punkty przestrzennie. Intuicja: prawdziwe wjazdy, zjazdy i krance badanego odcinka powinny zbierac wiele startow albo koncow.

In [ ]:
valid_trip_ids = trip_stats.loc[trip_stats["usable_for_endpoint_detection"], "trip_uid"]

start_points = trip_stats[trip_stats["usable_for_endpoint_detection"]][[
    "trip_uid", "vehicle_id", "start_time", "start_lat", "start_lon", "origin_region"
]].rename(columns={
    "start_time": "event_time",
    "start_lat": "lat",
    "start_lon": "lon",
    "origin_region": "region",
})
start_points["endpoint_type"] = "origin"

end_points = trip_stats[trip_stats["usable_for_endpoint_detection"]][[
    "trip_uid", "vehicle_id", "end_time", "end_lat", "end_lon", "destination_region"
]].rename(columns={
    "end_time": "event_time",
    "end_lat": "lat",
    "end_lon": "lon",
    "destination_region": "region",
})
end_points["endpoint_type"] = "destination"

endpoints = pd.concat([start_points, end_points], ignore_index=True)
endpoints.head()

In [ ]:
earth_radius_m = 6_371_000
eps_rad = PARAMS["endpoint_cluster_eps_m"] / earth_radius_m
coords_rad = np.radians(endpoints[["lat", "lon"]].to_numpy())

clusterer = DBSCAN(
    eps=eps_rad,
    min_samples=PARAMS["endpoint_cluster_min_samples"],
    metric="haversine",
)
endpoints["endpoint_cluster"] = clusterer.fit_predict(coords_rad)

cluster_stats = (
    endpoints[endpoints["endpoint_cluster"] >= 0]
    .groupby("endpoint_cluster")
    .agg(
        lat=("lat", "mean"),
        lon=("lon", "mean"),
        events=("trip_uid", "size"),
        origins=("endpoint_type", lambda s: (s == "origin").sum()),
        destinations=("endpoint_type", lambda s: (s == "destination").sum()),
        regions=("region", "nunique"),
    )
    .reset_index()
    .sort_values("events", ascending=False)
)

cluster_stats.head(20)

## 9. Krance obszaru, POI z OSM i nazwy klastrow

Endpoint moze byc wiarygodny, jesli lezy w duzym klastrze, na krancu badanego obszaru albo blisko istotnego POI. Na razie POI wpisujemy recznie jako szkielet. Pozniej mozna je uzupelnic z OSM lub z listy biznesowej.

In [ ]:
manual_poi = pd.DataFrame([
    # Przyklady do uzupelnienia lub nadpisania nazw z OSM:
    # {"poi_name": "Westfield Arkadia", "lat": 52.2579, "lon": 20.9847, "poi_type": "shopping_centre", "source": "manual"},
    # {"poi_name": "Metro Marymont", "lat": 52.2713, "lon": 20.9717, "poi_type": "transport", "source": "manual"},
])

# OSM pobieramy tylko raz i zapisujemy do cache. Jesli nie ma internetu, notebook nadal dziala z pustym zestawem POI.
osm_tags = {
    "highway": ["motorway_junction", "bus_stop"],
    "amenity": ["parking", "parking_entrance", "fuel", "charging_station", "bus_station", "hospital", "university"],
    "shop": ["mall", "supermarket", "department_store"],
    "railway": ["station", "halt", "tram_stop", "subway_entrance"],
    "public_transport": ["station", "stop_position", "platform"],
    "tourism": ["hotel"],
}

if OSM_POI_CACHE.exists():
    osm_poi = gpd.read_file(OSM_POI_CACHE)
elif ox is not None:
    try:
        west, south, east, north = df["lon"].min(), df["lat"].min(), df["lon"].max(), df["lat"].max()
        buffer_deg = PARAMS["osm_bbox_buffer_deg"]
        bbox = (west - buffer_deg, south - buffer_deg, east + buffer_deg, north + buffer_deg)
        osm_poi = ox.features_from_bbox(bbox, tags=osm_tags).reset_index()
        OSM_POI_CACHE.parent.mkdir(parents=True, exist_ok=True)
        osm_poi.to_file(OSM_POI_CACHE, driver="GeoJSON")
    except Exception as exc:
        print(f"OSM POI download failed: {exc}")
        osm_poi = gpd.GeoDataFrame(columns=["name", "geometry"], geometry="geometry", crs="EPSG:4326")
else:
    osm_poi = gpd.GeoDataFrame(columns=["name", "geometry"], geometry="geometry", crs="EPSG:4326")

if len(osm_poi) > 0:
    osm_poi = osm_poi.to_crs(epsg=4326)
    osm_poi_points = osm_poi.copy()
    osm_poi_points["geometry"] = osm_poi_points.geometry.representative_point()

    if {"poi_name", "poi_type"}.issubset(osm_poi_points.columns):
        osm_poi_points["poi_name"] = osm_poi_points["poi_name"].fillna("OSM object")
        osm_poi_points["poi_type"] = osm_poi_points["poi_type"].fillna("osm")
        osm_poi_points["source"] = osm_poi_points.get("source", "osm")
    else:
        osm_poi_points["poi_name"] = osm_poi_points.get("name", pd.Series(index=osm_poi_points.index, dtype="object"))
        osm_poi_points["poi_name"] = osm_poi_points["poi_name"].fillna(
            osm_poi_points.get("ref", pd.Series(index=osm_poi_points.index, dtype="object"))
        )
        osm_poi_points["poi_name"] = osm_poi_points["poi_name"].fillna("OSM object")
        osm_poi_points["poi_type"] = (
            osm_poi_points.get("amenity", pd.Series(index=osm_poi_points.index, dtype="object"))
            .fillna(osm_poi_points.get("shop", pd.Series(index=osm_poi_points.index, dtype="object")))
            .fillna(osm_poi_points.get("tourism", pd.Series(index=osm_poi_points.index, dtype="object")))
            .fillna(osm_poi_points.get("leisure", pd.Series(index=osm_poi_points.index, dtype="object")))
            .fillna(osm_poi_points.get("railway", pd.Series(index=osm_poi_points.index, dtype="object")))
            .fillna(osm_poi_points.get("highway", pd.Series(index=osm_poi_points.index, dtype="object")))
            .fillna(osm_poi_points.get("public_transport", pd.Series(index=osm_poi_points.index, dtype="object")))
            .fillna("osm")
        )
        osm_poi_points["source"] = "osm"

    poi_from_osm = pd.DataFrame({
        "poi_name": osm_poi_points["poi_name"],
        "poi_type": osm_poi_points["poi_type"],
        "source": osm_poi_points["source"],
        "lat": osm_poi_points.geometry.y,
        "lon": osm_poi_points.geometry.x,
    })
else:
    poi_from_osm = pd.DataFrame(columns=["poi_name", "poi_type", "source", "lat", "lon"])

poi = pd.concat([manual_poi, poi_from_osm], ignore_index=True)
poi.head(20)


In [ ]:
points_gdf = gpd.GeoDataFrame(
    endpoints,
    geometry=gpd.points_from_xy(endpoints["lon"], endpoints["lat"]),
    crs="EPSG:4326",
).to_crs(epsg=2180)

minx, miny, maxx, maxy = points_gdf.total_bounds
x = points_gdf.geometry.x
y = points_gdf.geometry.y
boundary_distances = pd.DataFrame({
    "west": x - minx,
    "east": maxx - x,
    "south": y - miny,
    "north": maxy - y,
})
points_gdf["distance_to_bbox_m"] = boundary_distances.min(axis=1)
points_gdf["boundary_side"] = boundary_distances.idxmin(axis=1)
points_gdf["near_boundary"] = points_gdf["distance_to_bbox_m"].le(PARAMS["boundary_distance_m"])

endpoints["distance_to_bbox_m"] = points_gdf["distance_to_bbox_m"].to_numpy()
endpoints["boundary_side"] = points_gdf["boundary_side"].to_numpy()
endpoints["near_boundary"] = points_gdf["near_boundary"].to_numpy()

if len(poi) > 0:
    poi_gdf = gpd.GeoDataFrame(
        poi,
        geometry=gpd.points_from_xy(poi["lon"], poi["lat"]),
        crs="EPSG:4326",
    ).to_crs(epsg=2180)
    nearest = gpd.sjoin_nearest(points_gdf, poi_gdf, how="left", distance_col="distance_to_poi_m")
    endpoints["nearest_poi"] = nearest["poi_name"].to_numpy()
    endpoints["distance_to_poi_m"] = nearest["distance_to_poi_m"].to_numpy()
    endpoints["near_poi"] = endpoints["distance_to_poi_m"].le(PARAMS["poi_distance_m"])
else:
    endpoints["nearest_poi"] = pd.NA
    endpoints["distance_to_poi_m"] = np.nan
    endpoints["near_poi"] = False

endpoints.head()

In [ ]:
def poi_priority(poi_type: str, poi_name: str) -> int:
    poi_type = str(poi_type).lower()
    poi_name = str(poi_name).lower()
    if poi_type in {"motorway_junction"}:
        return 0
    if poi_type in {"fuel", "charging_station"}:
        return 1
    if poi_type in {"mall", "supermarket", "department_store"}:
        return 2
    if poi_type in {"station", "halt", "subway_entrance", "bus_station"}:
        return 3
    if poi_type in {"parking", "parking_entrance"}:
        return 4
    if poi_type in {"hospital", "university", "hotel"}:
        return 5
    if "osm object" in poi_name:
        return 9
    return 6


if len(poi) > 0:
    poi_for_names = poi.copy()
    poi_for_names["name_priority"] = [
        poi_priority(t, n) for t, n in zip(poi_for_names["poi_type"], poi_for_names["poi_name"])
    ]
    poi_gdf_for_names = gpd.GeoDataFrame(
        poi_for_names,
        geometry=gpd.points_from_xy(poi_for_names["lon"], poi_for_names["lat"]),
        crs="EPSG:4326",
    ).to_crs(epsg=2180)
else:
    poi_gdf_for_names = gpd.GeoDataFrame(
        columns=["poi_name", "poi_type", "source", "name_priority", "geometry"],
        geometry="geometry",
        crs="EPSG:2180",
    )

if len(cluster_stats) > 0:
    cluster_gdf = gpd.GeoDataFrame(
        cluster_stats,
        geometry=gpd.points_from_xy(cluster_stats["lon"], cluster_stats["lat"]),
        crs="EPSG:4326",
    ).to_crs(epsg=2180)
else:
    cluster_gdf = gpd.GeoDataFrame(cluster_stats, geometry=[], crs="EPSG:2180")


def best_nearby_poi(point, max_distance_m=600):
    if len(poi_gdf_for_names) == 0:
        return pd.Series({
            "poi_name": pd.NA,
            "poi_type": pd.NA,
            "source": pd.NA,
            "distance_to_named_poi_m": np.nan,
        })
    distances = poi_gdf_for_names.geometry.distance(point)
    candidates = poi_gdf_for_names.loc[distances.le(max_distance_m), [
        "poi_name", "poi_type", "source", "name_priority"
    ]].copy()
    if len(candidates) == 0:
        nearest_idx = distances.idxmin()
        nearest = poi_gdf_for_names.loc[nearest_idx]
        return pd.Series({
            "poi_name": nearest["poi_name"],
            "poi_type": nearest["poi_type"],
            "source": nearest["source"],
            "distance_to_named_poi_m": float(distances.loc[nearest_idx]),
        })
    candidates["distance_to_named_poi_m"] = distances.loc[candidates.index]
    candidates = candidates.sort_values(["name_priority", "distance_to_named_poi_m"])
    best = candidates.iloc[0]
    return pd.Series({
        "poi_name": best["poi_name"],
        "poi_type": best["poi_type"],
        "source": best["source"],
        "distance_to_named_poi_m": float(best["distance_to_named_poi_m"]),
    })


if len(cluster_gdf) > 0:
    cluster_names = pd.concat(
        [
            cluster_stats.reset_index(drop=True),
            cluster_gdf.geometry.apply(best_nearby_poi).reset_index(drop=True),
        ],
        axis=1,
    )
else:
    cluster_names = cluster_stats.copy()
    cluster_names["poi_name"] = pd.NA
    cluster_names["poi_type"] = pd.NA
    cluster_names["source"] = pd.NA
    cluster_names["distance_to_named_poi_m"] = np.nan

cluster_names["poi_label"] = cluster_names["poi_name"].astype("string")
unnamed_poi = cluster_names["poi_label"].str.lower().eq("osm object")
cluster_names.loc[unnamed_poi, "poi_label"] = cluster_names.loc[unnamed_poi, "poi_type"].astype("string")

cluster_names["cluster_label"] = np.where(
    cluster_names["poi_label"].notna(),
    (
            "cluster_" + cluster_names["endpoint_cluster"].astype(str)
            + "_near_" + cluster_names["poi_label"].astype(str)
            + "_" + cluster_names["distance_to_named_poi_m"].round(0).astype("Int64").astype(str) + "m"
    ),
    "cluster_" + cluster_names["endpoint_cluster"].astype(str),
)

cluster_label_map = cluster_names.set_index("endpoint_cluster")["cluster_label"].to_dict()
cluster_names.sort_values("events", ascending=False).head(30)


In [ ]:
def endpoint_reliability(row):
    if row["endpoint_cluster"] >= 0:
        return "cluster"
    if row["near_boundary"]:
        return "boundary"
    if row["near_poi"]:
        return "poi"
    return "uncertain"


endpoints["endpoint_reliability"] = endpoints.apply(endpoint_reliability, axis=1)
endpoints["terminal_id"] = "uncertain"
endpoints["terminal_name"] = "uncertain"

boundary_mask = endpoints["near_boundary"]
endpoints.loc[boundary_mask, "terminal_id"] = "boundary_" + endpoints.loc[boundary_mask, "boundary_side"].astype(str)
endpoints.loc[boundary_mask, "terminal_name"] = endpoints.loc[boundary_mask, "terminal_id"]

poi_mask = endpoints["near_poi"]
endpoints.loc[poi_mask, "terminal_id"] = "poi_" + endpoints.loc[poi_mask, "nearest_poi"].astype(str)
endpoints.loc[poi_mask, "terminal_name"] = endpoints.loc[poi_mask, "terminal_id"]

cluster_mask = endpoints["endpoint_cluster"].ge(0)
endpoints.loc[cluster_mask, "terminal_id"] = "cluster_" + endpoints.loc[cluster_mask, "endpoint_cluster"].astype(str)
endpoints.loc[cluster_mask, "terminal_name"] = endpoints.loc[cluster_mask, "endpoint_cluster"].map(cluster_label_map)
endpoints["terminal_name"] = endpoints["terminal_name"].fillna(endpoints["terminal_id"])

endpoints["endpoint_reliability"].value_counts()


In [ ]:
endpoint_points_for_names = gpd.GeoDataFrame(
    endpoints,
    geometry=gpd.points_from_xy(endpoints["lon"], endpoints["lat"]),
    crs="EPSG:4326",
).to_crs(epsg=2180)

endpoint_nearest = endpoint_points_for_names.geometry.apply(lambda point: best_nearby_poi(point, max_distance_m=400))
endpoints["endpoint_nearest_poi_name"] = endpoint_nearest["poi_name"].to_numpy()
endpoints["endpoint_nearest_poi_type"] = endpoint_nearest["poi_type"].to_numpy()
endpoints["endpoint_nearest_poi_distance_m"] = endpoint_nearest["distance_to_named_poi_m"].to_numpy()

endpoints["endpoint_nearest_poi_label"] = endpoints["endpoint_nearest_poi_name"].astype("string")
unnamed_endpoint_poi = endpoints["endpoint_nearest_poi_label"].str.lower().eq("osm object")
endpoints.loc[unnamed_endpoint_poi, "endpoint_nearest_poi_label"] = (
    endpoints.loc[unnamed_endpoint_poi, "endpoint_nearest_poi_type"].astype("string")
)

endpoints["endpoint_label"] = np.where(
    endpoints["endpoint_reliability"].eq("cluster"),
    endpoints["terminal_name"],
    np.where(
        endpoints["endpoint_nearest_poi_label"].notna(),
        endpoints["endpoint_type"] + "_near_" + endpoints["endpoint_nearest_poi_label"].astype(str),
        endpoints["terminal_name"],
    ),
)

endpoints[[
    "trip_uid", "endpoint_type", "endpoint_label", "endpoint_reliability",
    "endpoint_nearest_poi_name", "endpoint_nearest_poi_type", "endpoint_nearest_poi_distance_m",
]].head(20)


## 10. Mapa endpointow

Ta mapa pomaga ocenic, czy klastry endpointow rzeczywiscie wypadaja na wjazdach, zjazdach i krancach analizowanego fragmentu S8.

In [ ]:
import html

m = folium.Map(location=[df["lat"].mean(), df["lon"].mean()], zoom_start=12, tiles="CartoDB positron")

colors = {
    "cluster": "green",
    "boundary": "blue",
    "poi": "purple",
    "uncertain": "red",
}

legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999; background: white; padding: 10px 12px; border: 1px solid #999; border-radius: 4px; font-size: 13px;">
  <b>Endpointy</b><br>
  <span style="color: green;">?</span> klaster endpointow<br>
  <span style="color: blue;">?</span> kraniec obszaru<br>
  <span style="color: purple;">?</span> blisko POI<br>
  <span style="color: red;">?</span> niepewny punkt<br>
  <span style="color: black;">?</span> najwieksze wiarygodne klastry
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

sample_endpoints = endpoints.sample(min(len(endpoints), 3000), random_state=42)
for _, row in sample_endpoints.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=3,
        color=colors[row["endpoint_reliability"]],
        fill=True,
        fill_opacity=0.65,
        tooltip=(
            f"{row.endpoint_type}<br>"
            f"reliability: {row.endpoint_reliability}<br>"
            f"cluster: {row.endpoint_cluster}<br>"
            f"terminal: {row.terminal_name}<br>"
            f"endpoint label: {row.endpoint_label}<br>"
            f"trip: {row.trip_uid}"
        ),
    ).add_to(m)

labelled_clusters = cluster_names.sort_values("events", ascending=False).head(25)
for _, row in labelled_clusters.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=6 + np.log1p(row["events"]),
        color="black",
        fill=False,
        weight=2,
        tooltip=(
            f"cluster: {row.endpoint_cluster}<br>"
            f"label: {row.cluster_label}<br>"
            f"events: {row.events}<br>"
            f"origins: {row.origins}<br>"
            f"destinations: {row.destinations}"
        ),
    ).add_to(m)

    short_label = str(row["cluster_label"]).replace("cluster_", "c")
    if len(short_label) > 42:
        short_label = short_label[:39] + "..."
    folium.Marker(
        location=[row["lat"], row["lon"]],
        icon=folium.DivIcon(
            html=f"""
            <div style="font-size: 11px; color: black; background: rgba(255,255,255,0.78); padding: 1px 3px; border-radius: 3px; border: 1px solid #666; white-space: nowrap;">
              {html.escape(short_label)}
            </div>
            """
        ),
    ).add_to(m)

ENDPOINT_MAP_FILE = Path("trip_endpoint_candidates_map.html")
m.save(ENDPOINT_MAP_FILE)
m


## 11. Przejazdy wiarygodne i pierwsza macierz OD

Na tym etapie bierzemy tylko przejazdy, ktore maja wiarygodny poczatek i koniec. To konserwatywne podejscie: lepiej stracic troche przejazdow niz zbudowac macierz z urwanych srodkow trasy.

In [ ]:
origin_terminals = endpoints[endpoints["endpoint_type"] == "origin"][[
    "trip_uid", "terminal_id", "terminal_name", "endpoint_label", "endpoint_reliability", "endpoint_cluster"
]].rename(columns={
    "terminal_id": "origin_terminal",
    "terminal_name": "origin_terminal_name",
    "endpoint_label": "origin_endpoint_label",
    "endpoint_reliability": "origin_reliability",
    "endpoint_cluster": "origin_cluster",
})

destination_terminals = endpoints[endpoints["endpoint_type"] == "destination"][[
    "trip_uid", "terminal_id", "terminal_name", "endpoint_label", "endpoint_reliability", "endpoint_cluster"
]].rename(columns={
    "terminal_id": "destination_terminal",
    "terminal_name": "destination_terminal_name",
    "endpoint_label": "destination_endpoint_label",
    "endpoint_reliability": "destination_reliability",
    "endpoint_cluster": "destination_cluster",
})

trip_od = (
    trip_stats
    .merge(origin_terminals, on="trip_uid", how="left")
    .merge(destination_terminals, on="trip_uid", how="left")
)

reliable_labels = {"cluster", "boundary", "poi"}
trip_od["reliable_od"] = (
        trip_od["valid_basic"]
        & trip_od["mode_class"].eq("car_like")
        & trip_od["origin_reliability"].isin(reliable_labels)
        & trip_od["destination_reliability"].isin(reliable_labels)
        & trip_od["origin_terminal"].ne(trip_od["destination_terminal"])
)

trip_od["reliable_od"].value_counts()

In [ ]:
reliable_trips = trip_od[trip_od["reliable_od"]].copy()

traffic_matrix = pd.crosstab(
    reliable_trips["origin_terminal"],
    reliable_trips["destination_terminal"],
)

traffic_matrix

In [ ]:
od_summary = (
    reliable_trips
    .groupby(["origin_terminal", "origin_terminal_name", "destination_terminal", "destination_terminal_name"])
    .agg(
        trips=("trip_uid", "size"),
        mean_duration_s=("duration_s", "mean"),
        median_duration_s=("duration_s", "median"),
        mean_distance_m=("distance_m", "mean"),
    )
    .reset_index()
    .sort_values("trips", ascending=False)
)

od_summary.head(30)

## 12. Prezentacja macierzy przejazdow

Poni?ej pokazujemy kilka komplementarnych widokow macierzy OD. Ka?dy odpowiada na troch? inne pytanie: tabela jest dobra do audytu liczb, heatmapa do szukania wzorcow, ranking do wskazania najwazniejszych relacji, a mapa do sprawdzenia, czy przeplywy maja sens przestrzenny.


In [ ]:
terminal_names = (
    endpoints[["terminal_id", "terminal_name"]]
    .drop_duplicates()
    .set_index("terminal_id")["terminal_name"]
    .to_dict()
)

traffic_matrix_named = traffic_matrix.rename(index=terminal_names, columns=terminal_names)
traffic_matrix_named


### Komentarz: tabela macierzy

Macierz pokazuje liczbe wiarygodnych przejazdow samochodopodobnych z terminala poczatkowego do terminala koncowego. Zera oznaczaja brak zaobserwowanych przejazdow w tej jednej godzinie danych, a nie dowod, ze relacja nie istnieje w ogole. Przy obecnej probce najbezpieczniej interpretowac najwieksze wartosci i relacje powtarzalne, a drobne liczby traktowac jako sygnal do walidacji.


In [ ]:
top_terminals = (
    reliable_trips["origin_terminal"].value_counts()
    .add(reliable_trips["destination_terminal"].value_counts(), fill_value=0)
    .sort_values(ascending=False)
    .head(20)
    .index
)

matrix_top = traffic_matrix.reindex(index=top_terminals, columns=top_terminals, fill_value=0)
matrix_top_named = matrix_top.rename(index=terminal_names, columns=terminal_names)

plt.figure(figsize=(12, 9))
sns.heatmap(matrix_top_named, cmap="YlOrRd", linewidths=0.3, linecolor="white")
plt.title("Macierz OD dla 20 najaktywniejszych terminali")
plt.xlabel("terminal docelowy")
plt.ylabel("terminal poczatkowy")
plt.xticks(rotation=60, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()


### Komentarz: heatmapa

Heatmapa pozwala szybko zobaczyc koncentracje ruchu: ciemniejsze pola to relacje z wieksza liczba przejazdow. Przy wielu terminalach pelna macierz robi sie malo czytelna, dlatego ograniczamy widok do najbardziej aktywnych punktow. To dobry widok do rozmowy biznesowej: od razu widac, ktore wjazdy/zjazdy dominuja w probce.


In [ ]:
od_summary_display = od_summary.copy()
od_summary_display["origin"] = od_summary_display["origin_terminal_name"].fillna(od_summary_display["origin_terminal"])
od_summary_display["destination"] = od_summary_display["destination_terminal_name"].fillna(
    od_summary_display["destination_terminal"])
od_summary_display["mean_duration_min"] = od_summary_display["mean_duration_s"] / 60
od_summary_display["median_duration_min"] = od_summary_display["median_duration_s"] / 60
od_summary_display["mean_distance_km"] = od_summary_display["mean_distance_m"] / 1000

od_summary_display[[
    "origin", "destination", "trips", "mean_duration_min", "median_duration_min", "mean_distance_km"
]].head(25)


In [ ]:
top_edges = od_summary_display.head(20).sort_values("trips")

plt.figure(figsize=(11, 8))
plt.barh(
    top_edges["origin"] + " -> " + top_edges["destination"],
    top_edges["trips"],
    color="#2b8cbe",
)
plt.title("Najczestsze relacje OD")
plt.xlabel("liczba przejazdow")
plt.ylabel("")
plt.tight_layout()


### Komentarz: ranking relacji

Ranking jest najprostszy do decyzyjnego odczytu: pokazuje, ktore pary terminali sa najwazniejsze w tej probce. W praktyce to dobry kandydat na liste relacji do recznej walidacji na mapie oraz do nadania docelowych, biznesowych nazw terminalom.


In [ ]:
row_sums = traffic_matrix.sum(axis=1).replace(0, np.nan)
od_probability_matrix = traffic_matrix.div(row_sums, axis=0).fillna(0)
od_probability_matrix_named = od_probability_matrix.rename(index=terminal_names, columns=terminal_names)

od_probability_matrix_named.round(3)


### Komentarz: macierz prawdopodobienstw

Macierz prawdopodobienstw normalizuje kazdy wiersz do sumy 1. Dla danego terminala startowego pokazuje, jak rozkladaja sie obserwowane cele. To jest przydatne, gdy chcemy modelowac wybor kierunku po wjezdzie na trase albo porownywac terminale o roznej liczbie obserwacji.


In [ ]:
terminal_points = (
    endpoints[endpoints["terminal_id"].isin(
        set(reliable_trips["origin_terminal"]) | set(reliable_trips["destination_terminal"]))]
    .groupby(["terminal_id", "terminal_name"], as_index=False)
    .agg(lat=("lat", "mean"), lon=("lon", "mean"), events=("trip_uid", "size"))
)
terminal_points = terminal_points.set_index("terminal_id")

flow_map = folium.Map(location=[df["lat"].mean(), df["lon"].mean()], zoom_start=12, tiles="CartoDB positron")

flow_edges = od_summary_display.head(60).copy()
flow_min, flow_max = flow_edges["trips"].min(), flow_edges["trips"].max()
flow_colormap = cm.LinearColormap(["#ffffcc", "#fd8d3c", "#800026"], vmin=flow_min, vmax=flow_max)
flow_colormap.caption = "Liczba przejazdow OD"

for _, row in flow_edges.iterrows():
    if row["origin_terminal"] not in terminal_points.index or row["destination_terminal"] not in terminal_points.index:
        continue
    origin = terminal_points.loc[row["origin_terminal"]]
    destination = terminal_points.loc[row["destination_terminal"]]
    width = 1 + 6 * np.log1p(row["trips"]) / np.log1p(flow_max)
    folium.PolyLine(
        locations=[[origin["lat"], origin["lon"]], [destination["lat"], destination["lon"]]],
        color=flow_colormap(row["trips"]),
        weight=width,
        opacity=0.65,
        tooltip=(
            f"{row.origin} -> {row.destination}<br>"
            f"przejazdy: {row.trips}<br>"
            f"mediana czasu: {row.median_duration_min:.1f} min<br>"
            f"sredni dystans: {row.mean_distance_km:.1f} km"
        ),
    ).add_to(flow_map)

for terminal_id, row in terminal_points.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=4 + np.log1p(row["events"]),
        color="black",
        fill=True,
        fill_color="white",
        fill_opacity=0.9,
        tooltip=f"{row.terminal_name}<br>events: {row.events}",
    ).add_to(flow_map)

flow_colormap.add_to(flow_map)
OD_FLOW_MAP_FILE = Path("od_flow_map.html")
flow_map.save(OD_FLOW_MAP_FILE)
flow_map


### Komentarz: mapa przeplywow

Mapa przeplywow jest najlepsza do kontroli przestrzennej: czy kierunki relacji sa logiczne, czy nie laczymy przypadkiem sasiednich zjazdow, i czy najwieksze przeplywy przebiegaja po sensownych odcinkach. Linie sa uproszczone jako polaczenia terminal-terminal, wiec nie pokazuja dokladnej trasy przejazdu; to wizualizacja macierzy, nie map matching.


## 13. Co mozna zrobic na podstawie macierzy przejazdow?

Na bazie macierzy OD mozna m.in.:

- wskazac najwazniejsze relacje wjazd-zjazd i priorytetyzowac je do dalszej walidacji,
- porownywac rozklady kierunkow miedzy godzinami, dniami roboczymi i weekendami,
- szacowac, ktore wezly lub zjazdy sa najwiekszymi generatorami/ruchu w badanej probce,
- budowac modele prawdopodobienstwa celu dla pojazdu startujacego z danego terminala,
- laczyc macierz z czasami przejazdu i wykrywac relacje o wysokim wolumenie oraz duzym opoznieniu,
- symulowac skutki zmian organizacji ruchu, jesli macierz zostanie skalibrowana do znanych pomiarow natezenia,
- identyfikowac brakujace albo niepewne terminale, gdy wiele przejazdow konczy sie poza wiarygodnymi punktami.

Najwazniejsze ograniczenie: obecna macierz opisuje obserwowane slady telefonow w jednej godzinie, nie pelne natezenie ruchu. Do uzycia operacyjnego trzeba ja skalibrowac lub przynajmniej porownac z niezaleznym pomiarem ruchu.


## 14. Co trzeba dopracowac

- Zweryfikowac na mapie, czy klastry endpointow pokrywaja sie z wjazdami, zjazdami i krancami danych.
- Dobrac `endpoint_cluster_eps_m` i `endpoint_cluster_min_samples` tak, aby nie laczyc sasiednich wezlow.
- Zweryfikowac POI z OSM i recznie poprawic nazwy najwazniejszych klastrow, jesli najblizszy obiekt OSM nie opisuje dobrze wezla.
- Rozroznic korek od postoju: obecnie logika jest prosta i wymaga walidacji na przykladowych trajektoriach.
- Dopracowac detekcje rowerow/outlierow na podstawie walidacji mapowej, ale na razie nie stroic hiperparametrow.
- Docelowo terminale powinny miec nazwy biznesowe, np. `S8_zachod`, `Marymoncka_wjazd`, `Wislotrasa_zjazd`, zamiast technicznych `cluster_12`.